In [9]:
from dotenv import load_dotenv
load_dotenv()

True

### Before Agent

In [14]:
forbidden_topics = {
    "cheating": ["답지", "정답 알려줘", "숙제 대신", "써줘", "베끼기"],
    "distraction": ["롤", "게임", "유튜브", "웹튠", "웃긴"],
    "harmful": ["담배", "술", "폭력", "싸움", "바보"]
}

In [5]:
from langchain.agents.middleware import before_agent

@before_agent(can_jump_to = ["end"])
def education_guardrail(state, runtime):
    """
    학생의 질문 의도를 파악하여 교육적이지 않거나 부정행위가 의심될 경우,
    LLM에게 질문을 넘기지 않고 교육적인 멘트로 즉시 교정합니다.
    """
    
    if not state["messages"]:
        return None
    
    last_message = state["messages"][-1]
    if last_message.type != "human":
        return None
    
    user_text = last_message.content

    for keyword in forbidden_topics["cheating"]:
        if keyword in user_text:
            return {
                "messages": [{
                    "role": "assistant",
                    "content": "스스로 고민해봐야 실력이 늘어요! 정답을 바로 알려드리는 대신, 힌트를 들릴까요?"
                }],
                "jump_to": "end"
            }

    for keyword in forbidden_topics["distraction"]:
        if keyword in user_text:
            return {
                "messages": [{
                    "role": "assistant",
                    "content": "지금은 공부에 집중할 시간이에요! 딴짓은 쉬는 시간에 하세요."
                }],
                "jump_to": "end"
            }

    for keyword in forbidden_topics["harmful"]:
        if keyword in user_text:
            return {
                "messages": [{
                    "role": "assistant",
                    "content": "부적절한 대화 주제입니다. 바르고 고운 말을 사용해주세요."
                }],
                "jump_to": "end"
            }
        
    return None

In [5]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-2.5-flash",
    middleware=[education_guardrail],
)

In [6]:
agent.invoke(
    {"messages": [{"role": "user", "content": "피타고라스 정리가 잘 이해가 안돼."}]},
)

{'messages': [HumanMessage(content='피타고라스 정리가 잘 이해가 안돼.', additional_kwargs={}, response_metadata={}, id='d7704061-1009-4472-b97f-d4a580cb9bf7'),
  AIMessage(content='피타고라스 정리가 어렵게 느껴질 수 있지만, 핵심만 짚으면 생각보다 간단하고 재미있어요! 차근차근 설명해 드릴게요.\n\n---\n\n### 피타고라스 정리, 이것만 알면 돼요!\n\n피타고라스 정리는 한 문장으로 요약하면 이렇습니다.\n\n**"직각삼각형에서 빗변의 제곱은 나머지 두 변의 제곱의 합과 같다."**\n\n이 문장을 하나씩 뜯어볼게요.\n\n---\n\n#### 1. 핵심은 \'직각삼각형\'입니다! (The key is \'right-angled triangles\'!)\n\n*   **가장 중요한 전제 조건이에요.** 피타고라스 정리는 **오직 직각삼각형에서만** 성립합니다. 다른 모양의 삼각형(예: 예각삼각형, 둔각삼각형)에서는 사용할 수 없어요.\n*   **직각삼각형이란?** 한 각의 크기가 정확히 **90도(°)**인 삼각형을 말합니다. 우리가 흔히 보는 네모 반듯한 모서리(책상 모서리, 방 코너 등)가 90도예요.\n\n#### 2. 변의 이름을 알아야 해요 (You need to know the names of the sides)\n\n직각삼각형에는 세 변이 있는데, 각각 특별한 이름이 있어요.\n\n*   **빗변 (Hypotenuse):**\n    *   직각과 **마주보는 변**입니다.\n    *   세 변 중에서 **가장 긴 변**입니다.\n    *   이 변을 우리가 공식에서 주로 \'c\'라고 부릅니다.\n*   **나머지 두 변 (Legs):**\n    *   직각을 이루는 두 변입니다.\n    *   이 두 변을 우리가 공식에서 주로 \'a\'와 \'b\'라고 부릅니다. (a와 b는 서로 바꿔도 상관없어요!)\n\n 

In [7]:
agent.invoke(
    {"messages": [{"role": "user", "content": "독후감을 대신 써줘"}]},
)

{'messages': [HumanMessage(content='독후감을 대신 써줘', additional_kwargs={}, response_metadata={}, id='94d169b7-0160-44a1-90aa-085f55e7a36e'),
  AIMessage(content='스스로 고민해봐야 실력이 늘어요! 정답을 바로 알려드리는 대신, 힌트를 들릴까요?', additional_kwargs={}, response_metadata={}, id='e077a51c-9d98-40e9-b331-9445164d1780', tool_calls=[], invalid_tool_calls=[])]}

In [9]:
agent.invoke(
    {"messages": [{"role": "user", "content": "웃긴 얘기해줘"}]},
)

{'messages': [HumanMessage(content='웃긴 얘기해줘', additional_kwargs={}, response_metadata={}, id='6683a296-7f9b-4dbc-9887-3e30280c9164'),
  AIMessage(content='지금은 공부에 집중할 시간이에요! 딴짓은 쉬는 시간에 하세요.', additional_kwargs={}, response_metadata={}, id='d92be24c-fcca-4bbc-a22a-e73f6f748340', tool_calls=[], invalid_tool_calls=[])]}

In [10]:
agent.invoke(
    {"messages": [{"role": "user", "content": "담배 피고 싶어"}]},
)

{'messages': [HumanMessage(content='담배 피고 싶어', additional_kwargs={}, response_metadata={}, id='92ca0265-c993-46a7-b9db-cd15161a86dd'),
  AIMessage(content='부적절한 대화 주제입니다. 바르고 고운 말을 사용해주세요.', additional_kwargs={}, response_metadata={}, id='fcba04fd-d7a5-4616-863b-309615362a9e', tool_calls=[], invalid_tool_calls=[])]}

---

### After Agent

In [13]:
from langchain.chat_models import init_chat_model

safety_model = init_chat_model("google_genai:gemini-2.5-flash")

In [6]:
from langchain.agents.middleware import after_agent
from langchain.messages import AIMessage

@after_agent
def answer_leakage_guardrail(state, runtime):
    """
    AI가 답변을 생성한 '직후', 사용자에게 보여주기 전에 내용을 검사.
    만약 AI가 문제의 정답을 직접적으로 말해버렸다면, 이를 감지하고 수정.
    """

    if not state["messages"]:
        return None
    
    last_message = state["messages"][-1]
    if not isinstance(last_message, AIMessage):
        return None
    
    auditor_prompt = f"""
당신은 엄격한 교육 감독관입니다.
다음 '튜터의 답변'을 확인하세요.
답변이 학생을 지도하지 않고 문제의 정답이나 전체 풀이를 직접적으로 제공하다면 'LEAKED'라고 답하세요.
답변이 적절한 힌트나 설명을 제공한다면 'SAFE'라고 답하세요.

튜터의 답변: {last_message.content}
"""
    
    print("last_message:", last_message.content)
    
    result = safety_model.invoke([{"role": "user", "content": auditor_prompt}])
    
    if "LEAKED" in result.content:
        print("[가드레일 발동] 정답 유출 감지됨! 답변을 수정합니다.")
        last_message.content = "앗, 제가 정답을 바로 말할 뻔했네요!"

    return None

In [19]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-2.5-flash",
    middleware=[answer_leakage_guardrail],
)

In [20]:
agent.invoke(
    {"messages": [{"role": "user", "content": "직각 삼각형 두 직각변의 길이가 3과 4라면 빗변의 길이가 뭐야? 정답 알려줘."}]},
)

last_message: 직각 삼각형에서 빗변의 길이는 피타고라스 정리를 사용하여 구할 수 있습니다. 피타고라스 정리는 다음과 같습니다:

$a^2 + b^2 = c^2$

여기서 $a$와 $b$는 직각변의 길이이고, $c$는 빗변의 길이입니다.

주어진 값은 $a = 3$이고 $b = 4$이므로:

$3^2 + 4^2 = c^2$
$9 + 16 = c^2$
$25 = c^2$

$c$를 구하기 위해 양변에 제곱근을 취하면:

$c = \sqrt{25}$
$c = 5$

따라서 빗변의 길이는 **5**입니다.
[가드레일 발동] 정답 유출 감지됨! 답변을 수정합니다.


{'messages': [HumanMessage(content='직각 삼각형 두 직각변의 길이가 3과 4라면 빗변의 길이가 뭐야? 정답 알려줘.', additional_kwargs={}, response_metadata={}, id='04793cab-c123-44a4-bdb5-9e7da8768535'),
  AIMessage(content='앗, 제가 정답을 바로 말할 뻔했네요!', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019c373e-882f-7ff0-b147-96a168cb6d4c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 28, 'output_tokens': 833, 'total_tokens': 861, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 657}})]}

In [7]:
from langchain.agents.middleware import after_agent
from langchain.messages import AIMessage, SystemMessage, HumanMessage

@after_agent
def answer_leakage_guardrail(state, runtime):
    """
    AI가 답변을 생성한 '직후', 사용자에게 보여주기 전에 내용을 검사.
    만약 AI가 문제의 정답을 직접적으로 말해버렸다면, 이를 감지하고 수정.
    """

    if not state["messages"]:
        return None
    
    last_message = state["messages"][-1]
    if not isinstance(last_message, AIMessage):
        return None
    
    auditor_prompt = f"""
당신은 엄격한 교육 감독관입니다.
다음 '튜터의 답변'을 확인하세요.
답변이 학생을 지도하지 않고 문제의 정답이나 전체 풀이를 직접적으로 제공하다면 'LEAKED'라고 답하세요.
답변이 적절한 힌트나 설명을 제공한다면 'SAFE'라고 답하세요.

튜터의 답변: {last_message.content}
"""
    
    print("last_message:", last_message.content)
    print("-" * 10)
    
    result = safety_model.invoke([{"role": "user", "content": auditor_prompt}])
    
    if "LEAKED" in result.content:
        
        original_question = state["messages"][-2].content if len(state["messages"]) >= 2 else "사용자 질문 알 수 없음"

        correction_prompt = f"""
당신은 친절한 AI 튜터입니다.

절대 정답을 직접 말하지 말고, 학생 스스로 생각할 수 있도록 유도하는 질문이나 핵심 개념(힌트)만 설명하세요.
말투는 친절하게 해주세요.

사용자 질문: {original_question}
"""
        corrected_response = safety_model.invoke([
            SystemMessage(content="당신은 소크라테스식 교육볍을 사용하는 튜터입니다."),
            HumanMessage(content=correction_prompt)
        ])

        last_message.content = corrected_response.content

    return None

In [22]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-2.5-flash",
    middleware=[answer_leakage_guardrail],
)

In [23]:
agent.invoke(
    {"messages": [{"role": "user", "content": "직각 삼각형 두 직각변의 길이가 3과 4라면 빗변의 길이가 뭐야? 정답 알려줘."}]},
)

last_message: 직각 삼각형의 두 직각변의 길이가 3과 4라면, 빗변의 길이는 피타고라스의 정리를 사용하여 구할 수 있습니다.

피타고라스의 정리: $a^2 + b^2 = c^2$
(여기서 $a$와 $b$는 직각변의 길이, $c$는 빗변의 길이입니다.)

계산:
$3^2 + 4^2 = c^2$
$9 + 16 = c^2$
$25 = c^2$
$c = \sqrt{25}$
$c = 5$

따라서 빗변의 길이는 **5**입니다.
----------


{'messages': [HumanMessage(content='직각 삼각형 두 직각변의 길이가 3과 4라면 빗변의 길이가 뭐야? 정답 알려줘.', additional_kwargs={}, response_metadata={}, id='b245bb07-3179-44a6-a4ee-8e2aa699cc40'),
  AIMessage(content='안녕하세요! 직각삼각형의 빗변 길이를 찾고 계시는군요. 정말 좋은 질문이에요.\n\n직각삼각형의 세 변의 길이 사이에는 아주 특별하고 중요한 관계가 있답니다. 혹시 그 관계를 설명해주는 유명한 수학적 원리나 공식이 떠오르는 것이 있으신가요?\n\n그 공식을 떠올리시면, 주어진 두 변의 길이를 이용해서 빗변의 길이를 직접 계산해볼 수 있을 거예요! 😊', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019c3746-ca71-77b0-8204-5192710086c0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 28, 'output_tokens': 733, 'total_tokens': 761, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 579}})]}

---

In [3]:
from langchain.agents.middleware import before_agent
import re

@before_agent
def student_safety_middleware(state, runtime):
    """
    학생의 전화번호나 이메일이 감지되면 마스킹 처리하여 안전을 확보
    """
    if not state["messages"]: return None
    last_message = state["messages"][-1]
    if state["messages"][-1].type != "human": return

    content = last_message.content
    original_content = last_message.content

    # 전화번호 패턴 (010-XXXX-XXXX 또는 010XXXXXXXX 등)
    phone_pattern = r'01[016789]-?[0-9]{3,4}-?[0-9]{4}'
    # 이메일 패턴
    email_pattern = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'

    is_redacted = False

    if re.search(phone_pattern, original_content):
        content = re.sub(phone_pattern, '<PHONE_REDACTED>', content)
        is_redacted = True

    if re.search(email_pattern, original_content):
        content = re.sub(phone_pattern, '<EMAIL_REDACTED>', content)
        is_redacted = True

    if is_redacted:
        print(f"[학생 보호] 개인정보가 감지되어 마스킹 처리했습니다.\n원본: {original_content}\n수정: {content}")
        last_message.content = content

    return None


In [1]:
ESCALITION_KEYWORDS = ["왕따", "괴롭힘", "우울해", "학교 폭력", "상담 선생님", "사람 불러줘"]

In [4]:
@before_agent(can_jump_to=["end"])
def counseling_escalation_middleware(state, runtime):
    """
    심리적 위기 상황이나 상담 요청이 감지되면 AI 답변을 멈추고 인간 상담사에게 알림을 보냅니다.
    """
    if not state["messages"]: return None
    last_message = state["messages"][-1]
    if state["messages"][-1].type != "human": return

    for keyword in ESCALITION_KEYWORDS:
        if keyword in last_message.content:
            print(f"[상담 이관] 심각한 고민/요청 감지: {keyword}")

            # Slack, Email 등으로 실제 상담 교사에게 알림
            
            return {
                "messages": [{
                    "role": "assistant",
                    "content": "학생, 많이 힘들었겠구나. 이 문제는 내가 답변하기보다는 전문 상담 선생님이 직접 듣고 도와주시는 것이 좋을 것 같아."
                }],
                "jump_to": "end"
            }
        
    return None


In [10]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-2.5-flash",
    middleware=[
        answer_leakage_guardrail,
        counseling_escalation_middleware,
        student_safety_middleware,
        education_guardrail
    ],
)

In [15]:
agent.invoke(
    {"messages": [{"role": "user", "content": "제 번호는 010-1234-5678입니다."}]},
)

[학생 보호] 개인정보가 감지되어 마스킹 처리했습니다.
원본: 제 번호는 010-1234-5678입니다.
수정: 제 번호는 <PHONE_REDACTED>입니다.
last_message: 네, 번호를 알려주셨군요. 다른 궁금한 점이나 도움이 필요하시면 말씀해주세요.
----------


{'messages': [HumanMessage(content='제 번호는 <PHONE_REDACTED>입니다.', additional_kwargs={}, response_metadata={}, id='3d98b7e1-7f82-4b51-a644-4896fee3d9d4'),
  AIMessage(content='네, 번호를 알려주셨군요. 다른 궁금한 점이나 도움이 필요하시면 말씀해주세요.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019c3bbd-e572-7080-971e-a0eae973c2f9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 911, 'total_tokens': 925, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 891}})]}

In [16]:
agent.invoke(
    {"messages": [{"role": "user", "content": "요즘 학교에서 우울해"}]},
)

[상담 이관] 심각한 고민/요청 감지: 우울해
last_message: 학생, 많이 힘들었겠구나. 이 문제는 내가 답변하기보다는 전문 상담 선생님이 직접 듣고 도와주시는 것이 좋을 것 같아.
----------


{'messages': [HumanMessage(content='요즘 학교에서 우울해', additional_kwargs={}, response_metadata={}, id='5577bdbf-cfbc-4c84-b600-d1d5efa4bf2d'),
  AIMessage(content='학생, 많이 힘들었겠구나. 이 문제는 내가 답변하기보다는 전문 상담 선생님이 직접 듣고 도와주시는 것이 좋을 것 같아.', additional_kwargs={}, response_metadata={}, id='374f4e21-9341-4d89-9e9e-4930d9f803f2', tool_calls=[], invalid_tool_calls=[])]}